# 09. 원장은 학습에 들어갈 모양인가 — 백테스트 원장 계약을 실행으로 잰다

> 2026-09-06 · 이동원 · 결론 문서:
> [평가파트 v1.1 §5](../../docs/평가파트/version1.1/동적기준선_ARIMA동반_백테스트원장.md) ·
> 검증기 [`supply/backtest_ledger.py`](../../supply/backtest_ledger.py) ·
> 시험 [`tests/test_backtest_ledger_contract.py`](../../tests/test_backtest_ledger_contract.py) ·
> 이슈 [#110](https://github.com/devlee328288/Alpha_Stack/issues/110)

---

## 이 노트북이 답하는 것

> **강민석 님이 PR #131 로 넣으신 원장(`signal_log` · `trade_log`)이, 데이터 파트가 #110 에서
> 여쭌 모양대로 나오는가 — 그리고 그것이 학습에 되먹일 수 있는 모양인가.**

PR 본문은 "칸 9개 추가 · HF 업로드 구현" 이라 적혀 있습니다. 옮겨 적기 전에 **실행으로**
셉니다. 정규식으로 센 칸 수가 한 번 틀렸기 때문입니다(23 이라 적었다가 22 였습니다).

| 잰 것 | 결과 |
|---|---|
| `signal_log` 칸 | **17** (8 + 9) |
| `trade_log` 칸 | **22** (13 + 9) — v1.0 문서의 "14칸" 은 오기 |
| 행 수 | 거래일 − 1 |
| `realized_return_5d` | 종가로 다시 계산한 값과 1e-16 안에서 일치 · 마지막 4행 NaN |
| HF 원장 2종 | 🔴 **아직 없다** (코드만 머지) |
| 선언된 환경(.venv)에서 import | 🔴 **안 된다** — `datasets` 를 최상단에서 import

---

## 0. 준비

`backtest_strategies.py` 는 최상단에서 `from datasets import Dataset` 을 합니다. `pyproject.toml` 이
`datasets` 를 일부러 안 넣어서(74행) 선언된 환경에서는 이 모듈이 import 되지 않고, 이 세션의
anaconda 파이썬에서는 pandas dist-info 가 둘이라 `datasets` 쪽이 깨집니다. 여기서는 import 가
실패할 때만 빈 대역을 끼워 `run_backtest` 만 읽습니다 — 업로드 경로는 부르지 않습니다.

In [1]:
import importlib
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

try:
    import datasets  # noqa: F401
    print("datasets: 진짜 모듈")
except Exception as e:  # noqa: BLE001
    stub = types.ModuleType("datasets")
    stub.Dataset = object
    sys.modules["datasets"] = stub
    print(f"datasets import 실패({type(e).__name__}) → 대역을 끼움")

bt = importlib.import_module("backtest.backtest_strategies")
from supply import backtest_ledger as bl  # noqa: E402

print("backtest_strategies:", Path(bt.__file__).relative_to(ROOT))

datasets import 실패(TypeError) → 대역을 끼움


backtest_strategies: backtest\backtest_strategies.py


---

## 1. 합성 시세 60거래일로 칸을 센다

예측 함수는 라벨과 확률을 함께 돌려주는 새 규약으로 씁니다. 확률은 상수가 아니게 둡니다.

In [2]:
days = pd.bdate_range("2024-06-03", periods=60)
rng = np.random.default_rng(0)
close = pd.Series(100 * np.cumprod(1 + rng.normal(0, 0.01, len(days))), index=days)
market = pd.DataFrame({"open": close.shift(1).fillna(close.iloc[0]), "close": close})


def predict(date, market_data):
    i = market_data.index.get_loc(date)
    return bl.LABELS[i % 3], {"p_up": 0.5, "p_flat": 0.3, "p_down": 0.2}


res = bt.run_backtest(market, days[0], days[-1], predict, strategy="A",
                      model_id="probe-v0", run_id="run_probe")
sig, trd = res["signal_log"], res["trade_log"]
print("signal_log:", sig.shape, "| trade_log:", trd.shape)
print("signal_log 칸이 계약과 같다:", list(sig.columns) == list(bl.SIGNAL_LOG_COLUMNS))
print("trade_log  칸이 계약과 같다:", list(trd.columns) == list(bl.TRADE_LOG_COLUMNS))
sig.head(3)

signal_log: (59, 17) | trade_log: (39, 22)
signal_log 칸이 계약과 같다: True
trade_log  칸이 계약과 같다: True


,prediction_date,execution_date,signal,consecutive_up,consecutive_down,requested_trade_ratio,actual_trade_value,position_ratio_after,code,p_up,p_flat,p_down,realized_return_5d,model_id,model_rev,run_id,cost_rate
0,2024-06-03,2024-06-04,상승,1,0,0.2,20.000000,0.200040,KOSPI200,0.5,0.3,0.2,0.004358,probe-v0,v0,run_probe,0.001
1,2024-06-04,2024-06-05,중립,0,0,0.0,0.000000,0.199829,KOSPI200,0.5,0.3,0.2,0.018801,probe-v0,v0,run_probe,0.001
2,2024-06-05,2024-06-06,하락,0,1,-0.2,20.016299,0.000851,KOSPI200,0.5,0.3,0.2,0.021905,probe-v0,v0,run_probe,0.001


### 행 수 · 체결일 · 실현수익률

마지막 거래일은 다음 날 체결이 없어 원장에 남지 않습니다 — 그래서 **거래일 − 1**.
`realized_return_5d` 는 `close[t+5] / close[t] − 1` 인데 t+5 가 자료 밖인 마지막 네 행이 NaN 입니다.

In [3]:
pos = {d: i for i, d in enumerate(days)}
pairs = zip(sig["prediction_date"], sig["execution_date"], strict=True)
gap = pd.Series([pos[e] - pos[p] for p, e in pairs])
mine = bl.recompute_realized_return(sig["prediction_date"], close)
both = sig["realized_return_5d"].notna() & mine.notna()
print(f"행 수 {len(sig)} = 거래일 {len(days)} − 1 :", len(sig) == len(days) - 1)
print("체결일 − 예측일 (거래일 수) 고유값:", sorted(gap.unique().tolist()))
print("실현수익률 NaN 행:", int(sig["realized_return_5d"].isna().sum()))
print("실현수익률 재계산 최대 차:", float((sig["realized_return_5d"] - mine).abs()[both].max()))

행 수 59 = 거래일 60 − 1 : True
체결일 − 예측일 (거래일 수) 고유값: [1]
실현수익률 NaN 행: 4
실현수익률 재계산 최대 차: 1.1102230246251565e-16


---

## 2. 검증기가 규격 그대로 나온 원장을 통과시키는가

In [4]:
r = bl.verify_execution_log(sig, close=close)
t = bl.verify_trade_log(trd, close=close)
print("signal_log problems:", r["problems"], "| warnings:", r["warnings"])
print("trade_log  problems:", t["problems"])
r["summary"]

signal_log problems: [] | warnings: ['확률이 전 행에서 상수다 — 학습에 정보가 없다 ([0.5, 0.3, 0.2])']
trade_log  problems: []


{'kind': 'signal',
 'holdout_start': '20240901',
 'missing_columns': [],
 'extra_columns': [],
 'prob_distinct_rows': 1,
 'holdout_rows': 0,
 'realized_nan_rows': 4,
 'holdout_peek_rows': 0,
 'signal_counts': {'상승': 20, '중립': 20, '하락': 19},
 'run_ids': 1,
 'model_ids': ['probe-v0']}

---

## 3. 실제 개발구간 파일로 — 강민석 님이 쓰시는 그 파일

HF `small/features_labels_kospi200_dev.csv` 를 **캐시에서** 읽습니다(다운로드 없음). 예측 함수는
저장소의 `predict_5d_after` 그대로 — 지금은 랜덤 예측이고 확률이 상수 0.33/0.34/0.33 입니다.

In [5]:
from huggingface_hub import try_to_load_from_cache

p = try_to_load_from_cache("qurious-quant/alphastack-krx-dev",
                           "small/features_labels_kospi200_dev.csv", repo_type="dataset")
df = pd.read_csv(p)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")
print(f"행 {len(df):,} · {df.index.min().date()} ~ {df.index.max().date()}")

real = bt.run_backtest(df, df.index[0], df.index[-1], bt.predict_5d_after,
                       strategy="A", model_id="random-v0", run_id="run_probe_real")
s = real["signal_log"]
print("signal_log:", s.shape, "| trade_log:", real["trade_log"].shape)
print("signal 분포:", s["signal"].value_counts().to_dict())
print("p_* 고유 조합:", s[["p_up", "p_flat", "p_down"]].drop_duplicates().values.tolist())

행 3,553 · 2010-03-30 ~ 2024-08-22


signal_log: (3552, 17) | trade_log: (1898, 22)
signal 분포: {'중립': 1247, '상승': 1186, '하락': 1119}
p_* 고유 조합: [[0.33, 0.34, 0.33]]


In [6]:
rr = bl.verify_execution_log(s, close=df["close"])
print("problems:", rr["problems"])
print("warnings:", rr["warnings"])
{k: v for k, v in rr["summary"].items() if k != "signal_counts"}

problems: []
warnings: ['확률이 전 행에서 상수다 — 학습에 정보가 없다 ([0.33, 0.34, 0.33])']


{'kind': 'signal',
 'holdout_start': '20240901',
 'missing_columns': [],
 'extra_columns': [],
 'prob_distinct_rows': 1,
 'holdout_rows': 0,
 'realized_nan_rows': 4,
 'holdout_peek_rows': 0,
 'run_ids': 1,
 'model_ids': ['random-v0']}

규격은 깨끗합니다. 경고 하나 — **확률이 전 행 상수**라 지금 원장을 되먹여도 배울 것이 없습니다.
그래서 되먹임 피처는 합의대로 2차이고, 1차는 **모양을 고정해 두는 것**까지입니다.
오준영 님 모델이 붙어 `predict_proba()` 가 세 칸을 채우는 순간 이 경고가 사라집니다.

---

## 4. 검증기가 잡아야 할 것을 잡는가 — 홀드아웃을 넘겨 돌리면

개발구간 예측이 봉인 구간(20240901~) 가격을 엿보는 두 가지 방식을 다 잡아야 합니다.
① 예측일 자체가 봉인 안 · ② 예측일은 개발구간인데 **t+5 종가**가 봉인 안.

In [7]:
d2 = pd.bdate_range("2024-08-12", periods=30)        # 08-12 ~ 09-20
c2 = pd.Series(100 * np.cumprod(1 + rng.normal(0, 0.01, len(d2))), index=d2)
m2 = pd.DataFrame({"open": c2.shift(1).fillna(c2.iloc[0]), "close": c2})
leak = bt.run_backtest(m2, d2[0], d2[-1], predict, strategy="A")["signal_log"]
v = bl.verify_execution_log(leak, close=c2)
print("예측일이 봉인 안인 행:", v["summary"]["holdout_rows"])
print("t+5 종가가 봉인 안인 행:", v["summary"]["holdout_peek_rows"])
for p_ in v["problems"]:
    print(" 🔴", p_)

예측일이 봉인 안인 행: 14
t+5 종가가 봉인 안인 행: 15
 🔴 예측일이 홀드아웃(20240901~) 안인 행: 14
 🔴 실현수익률이 홀드아웃 종가를 엿본 행: 15


---

## 5. HF 에 원장이 있나 — 없다

PR #131 은 `run_cost_sensitivity.py` 에 업로드 코드를 넣었습니다. 서버를 봅니다.

In [8]:
from huggingface_hub import HfApi

ds = sorted(d.id for d in HfApi().list_datasets(author="qurious-quant"))
for x in ds:
    print(" ", x)
print("\n원장 2종:", {x: (x in ds) for x in (bl.HF_EXECUTION_LOG, bl.HF_TRADE_LOG)})

  qurious-quant/alphastack-backtest-kospi200
  qurious-quant/alphastack-backtest-results
  qurious-quant/alphastack-breakeven-cost
  qurious-quant/alphastack-cost-sensitivity
  qurious-quant/alphastack-dart
  qurious-quant/alphastack-krx-dev

원장 2종: {'qurious-quant/alphastack-backtest-execution-log': False, 'qurious-quant/alphastack-backtest-trade-log': False}


---

## 6. 결론

| 물음 | 답 |
|---|---|
| 칸 아홉이 들어왔나 | ✅ `signal_log` 17 · `trade_log` 22 · 계약 상수와 순서까지 같다 |
| 값이 규격에 맞나 | ✅ 체결일은 다음 거래일 · 실현수익률은 종가 재계산과 일치 · 홀드아웃 행 0 |
| 학습에 배울 것이 있나 | ⚠️ 아직 — 확률이 상수(랜덤 예측). 모델이 붙으면 사라지는 경고 |
| HF 에 올라갔나 | 🔴 아니다 — 코드만. #110 에 실행을 요청 |
| 선언된 환경에서 도나 | 🔴 아니다 — `datasets` 최상단 import. 함수 안으로 옮기거나 `HfApi.upload_file` (#110) |

원장이 올라오면 `scripts/verify_backtest_ledger.py` 로 감싸 반출 게이트에 등록하고, `daily_price` ·
`adj_close` · 라벨과의 조인 정문을 `supply/` 에 냅니다.